# DENDRA: training a biologically inspired LLM (Kaggle)

This notebook trains **DENDRA** and a **GPT baseline** of the same size on the same data with the same tokenizer, then compares them.

**How to run on Kaggle**
1. *File -> Import Notebook*, choose this `.ipynb` file.
2. Right panel: *Session options -> Accelerator*, choose **GPU T4 x2** or **GPU P100**.
3. Turn on **Internet** (requires a phone-verified Kaggle account) to download the data.
4. *Run All*. Results, checkpoints and the tokenizer are saved to `/kaggle/working`.

**Default data:** `nampdn-ai/tinystories-vietnamese`, simple short stories translated into Vietnamese, well suited to small models. If downloading fails, the notebook falls back to Vietnamese Wikipedia, then Tiny Shakespeare.

Everything is set in the **Configuration** cell below.

In [ ]:
# ==================== CONFIGURATION ====================
import os
QUICK_TEST = os.environ.get("DENDRA_QUICK", "0") == "1"   # very fast smoke test (CPU)

CFG = dict(
    dataset="tinystories_vi",   # "tinystories_vi" | "wiki_vi" | "tinystories_en" | "shakespeare"
    max_docs=200_000,           # number of documents to download (increase for more data)
    vocab_size=8192,            # size of the BPE tokenizer trained on the data
    ctx=256,                    # context length (tokens)
    batch=32,
    steps=4000,                 # number of steps for EACH model
    lr=1e-3, warmup=200, weight_decay=0.1,
    eval_every=250, eval_batches=20,
    log_every=20,               # print progress every 20 steps (with speed and time remaining)
    max_minutes=60,             # time limit for EACH model (None = no limit)
    # --- DENDRA ---
    d=384, regions=6, heads=6, n_e=576, branches=4, T=4,
    fb_cycles=1,                # 2 = enable top-down feedback passes (twice as slow)
    spike_in=False,             # True = ON/OFF spike-encoded dendritic input (more additions instead of multiplications)
    neuromod=True, rate_target=0.15, homeo_w=1.0, reflex_w=0.3,
    # --- GPT baseline ---
    compare_gpt=True, gpt_layers=6,
    parallel_gpus=True,         # with 2 GPUs (T4 x2): DENDRA on GPU 0, GPT on GPU 1, at the same time
    share_free_gpu=True,        # GPT finishes first -> DENDRA also uses GPU 1 (each batch split in two)
    compile=True,               # torch.compile during training (fused ops, ~2x faster); falls back automatically on error
    bench_compile=True,         # also measure the "fast + compile" mode in the inference benchmark
    seed=0,
)
if QUICK_TEST:
    CFG.update(dataset="shakespeare", max_docs=2000, vocab_size=512, ctx=64, batch=8, steps=30,
               eval_every=15, eval_batches=2, log_every=5, compile=False, bench_compile=False, d=64, regions=2, heads=2, n_e=96, gpt_layers=2, warmup=5)
OUT = "/kaggle/working" if os.path.isdir("/kaggle/working") else "./dendra_out"
os.makedirs(OUT, exist_ok=True)
CFG

In [ ]:
import sys, subprocess, importlib
for pkg in ["datasets", "tokenizers"]:
    if importlib.util.find_spec(pkg) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg])

import math, time, json, random, re
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cuda":
    cap = torch.cuda.get_device_capability()
    AMP_DTYPE = torch.bfloat16 if cap[0] >= 8 else torch.float16
    print(f"GPUs: {torch.cuda.device_count()} ({torch.cuda.get_device_name(0)}) | AMP: {AMP_DTYPE}")
else:
    AMP_DTYPE = None
    print("No GPU, running on CPU (very slow, for testing only).")
torch.backends.cuda.matmul.allow_tf32 = True; torch.backends.cudnn.allow_tf32 = True
torch.manual_seed(CFG["seed"]); random.seed(CFG["seed"]); np.random.seed(CFG["seed"])
print("torch", torch.__version__)

## 1. Data

In [ ]:
VI_CHARS = set("ăâđêôơưạảấầẩẫậắằẳẵặẹẻẽếềểễệỉịọỏốồổỗộớờởỡợụủứừửữựỳỵỷỹáàãéèíìóòõúùýĂÂĐÊÔƠƯ")   # Vietnamese-specific letters (used to find the Vietnamese text column)

def _flatten(ex, prefix=""):
    out = {}
    for k, v in ex.items():
        if isinstance(v, dict):
            out.update(_flatten(v, f"{prefix}{k}."))
        elif isinstance(v, str):
            out[f"{prefix}{k}"] = v
    return out

def _get(ex, col):
    return _flatten(ex).get(col, "")

def pick_text_column(examples, want_vi=True):
    # Pick the text column: prefer the column with the most Vietnamese characters (the dataset has English and Vietnamese)
    scores = {}
    for ex in examples:
        for k, v in _flatten(ex).items():
            vi = sum(c in VI_CHARS for c in v)
            scores.setdefault(k, [0, 0])
            scores[k][0] += vi; scores[k][1] += len(v)
    if want_vi:
        return max(scores, key=lambda k: (scores[k][0], scores[k][1]))
    return max(scores, key=lambda k: scores[k][1])

def load_hf(name, config=None, want_vi=True, n=CFG["max_docs"]):
    from datasets import load_dataset
    tries = [dict(), dict(data_files={"train": "*.parquet"}), dict(data_files={"train": "**/*.parquet"})]
    err = None
    for kw in tries:   # some repos contain several kinds of parquet files; try them in turn
        try:
            ds = load_dataset(name, config, split="train", streaming=True, **kw)
            it = iter(ds)
            head = [next(it) for _ in range(20)]
            break
        except Exception as e:
            err = e; print(f"  attempt {kw or 'default'} failed: {type(e).__name__}: {str(e)[:150]}")
    else:
        raise err
    col = pick_text_column(head, want_vi)
    print(f"{name}: using column '{col}'. Example:", _get(head[0], col)[:200].replace("\n", " "))
    docs = [_get(e, col) for e in head]
    for ex in it:
        if len(docs) >= n: break
        t = _get(ex, col)
        if t and len(t) > 20: docs.append(t)
    return docs

def load_shakespeare():
    import urllib.request
    url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
    path = os.path.join(OUT, "shakespeare.txt")
    if not os.path.exists(path):
        urllib.request.urlretrieve(url, path)
    text = open(path, encoding="utf-8").read()
    return [p for p in text.split("\n\n") if p.strip()][:CFG["max_docs"]]

SOURCES = {
    "tinystories_vi": lambda: load_hf("nampdn-ai/tinystories-vietnamese", want_vi=True),
    "wiki_vi": lambda: load_hf("wikimedia/wikipedia", "20231101.vi", want_vi=True),
    "tinystories_en": lambda: load_hf("roneneldan/TinyStories", want_vi=False),
    "shakespeare": load_shakespeare,
}
order = [CFG["dataset"]] + [k for k in ["tinystories_vi", "wiki_vi", "shakespeare"] if k != CFG["dataset"]]
docs = None
for name in order:
    try:
        t0 = time.time(); docs = SOURCES[name](); DATA_NAME = name
        print(f"Loaded {len(docs):,} documents from '{name}' in {time.time()-t0:.0f}s"); break
    except Exception as e:
        print(f"Could not load '{name}': {type(e).__name__}: {str(e)[:200]}")
assert docs, "No dataset could be loaded. Check that Internet is enabled in the Kaggle settings."
print("Total characters:", f"{sum(map(len, docs)):,}")

In [ ]:
# Byte-level BPE tokenizer trained on the data itself (better for Vietnamese than the GPT-2 tokenizer)
from tokenizers import ByteLevelBPETokenizer
random.shuffle(docs)
n_val = max(1, int(0.02 * len(docs)))
val_docs, train_docs = docs[:n_val], docs[n_val:]

tok = ByteLevelBPETokenizer()
t0 = time.time()
tok.train_from_iterator(train_docs[:100_000], vocab_size=CFG["vocab_size"], min_frequency=2, special_tokens=["<|eos|>"])
EOS = tok.token_to_id("<|eos|>")
V = tok.get_vocab_size()
os.makedirs(f"{OUT}/tokenizer", exist_ok=True); tok.save_model(f"{OUT}/tokenizer")
print(f"Tokenizer: {V} tokens, trained in {time.time()-t0:.0f}s")

def encode_docs(ds):
    ids = []
    for i in range(0, len(ds), 10_000):
        for e in tok.encode_batch(ds[i:i+10_000]):
            ids.extend(e.ids); ids.append(EOS)
    return np.array(ids, dtype=np.int32)

train_ids, val_ids = encode_docs(train_docs), encode_docs(val_docs)
print(f"Training tokens: {len(train_ids):,} | validation: {len(val_ids):,} | "
      f"{sum(map(len, train_docs))/len(train_ids):.2f} characters/token")
_s = "Ngày xửa ngày xưa, có một chú mèo nhỏ."   # Vietnamese: 'Once upon a time, there was a little cat.'
print(f"Sample sentence '{_s}' -> {len(tok.encode(_s).ids)} tokens")
train_t, val_t = torch.from_numpy(train_ids).long(), torch.from_numpy(val_ids).long()

def get_batch(src, bs, gen=None, dev=None):
    dev = dev or device
    ix = torch.randint(len(src) - CFG["ctx"] - 1, (bs,), generator=gen)
    x = torch.stack([src[i:i+CFG["ctx"]] for i in ix])
    y = torch.stack([src[i+1:i+CFG["ctx"]+1] for i in ix])
    return x.to(dev, non_blocking=True), y.to(dev, non_blocking=True)

## 2. Models

* **DENDRA**: dendritic LIF neurons, Dale's-law E/I microcircuits, Hebbian-STDP memory instead of attention, neuromodulators (noradrenaline, acetylcholine), L6 feedback, a reflex path, homeostasis.
* **GPT**: a standard Transformer (pre-LN, causal attention) as the baseline.

In [ ]:
"""
DENDRA: a language model built from biological neuron mechanisms.

Main components
  LIF neuron + refractory period -> DendriticSpikingLayer (0/1 spikes, soft reset, adaptive threshold)
  Nonlinear dendrites            -> each neuron has B branches, each with its own local nonlinearity
  E (80%) / I (20%) populations  -> Dale's law: E->I and I->E weights are positive; I subtracts from E (soft k-WTA)
  Hebb + STDP + NMDA             -> HebbianSTDP: S <- lam*S + w * v k^T  (replaces attention)
  Neuromodulators (NE, ACh)      -> Neuromodulators: global gain and memory-write gate
  Feedback connections (L6)      -> several up/down passes, higher regions send signals down
  Spinal reflex                  -> early-exit head after region 1
  Homeostasis                    -> loss keeping the firing rate near a target (in the training script)

Three computation modes for the same weights (outputs match, max error < 1e-5):
  forward()      parallel over time, used for training
  prefill()      reads a prompt chunk by chunk, O(n * C)
  step_token()   generates one token with a constant-size state, O(1) memory per token
"""
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

# ---------------------------------------------------------------- spikes + surrogate gradient
class _Spike(torch.autograd.Function):
    """All-or-none: spike = 1 when the membrane potential crosses threshold. Surrogate gradient: fast sigmoid."""

    @staticmethod
    def forward(ctx, x, alpha):
        ctx.save_for_backward(x)
        ctx.alpha = alpha
        return (x > 0).to(x.dtype)

    @staticmethod
    def backward(ctx, g):
        (x,) = ctx.saved_tensors
        return g / (1.0 + ctx.alpha * x.abs()) ** 2, None


def spike(x, alpha=2.0):
    return _Spike.apply(x, alpha)


class DivisiveNorm(nn.Module):
    """Divisive normalization by an interneuron pool: divide by the mean activity energy."""

    def __init__(self, d, eps=1e-5):
        super().__init__()
        self.g = nn.Parameter(torch.ones(d))
        self.eps = eps

    def forward(self, x):
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps) * self.g


# ---------------------------------------------------------------- L2/3: Hebbian/STDP synapses
class HebbianSTDP(nn.Module):
    """
    Fast-weight memory that replaces attention.
      S_t = lam * S_{t-1} + w_t * v_t k_t^T        (Hebb: pre and post active together -> stronger synapse)
      o_t = S_t q_t
    - lam < 1: synaptic decay; each head has its own time constant (4..256 tokens), learned.
    - Parallel form: o = ((Q K^T) * D * w) V with D_ij = lam^(i-j) for j <= i. This is a causal STDP window
      (pre-before-post strengthens, decaying with distance), so no position embedding is needed.
    - w_t = NMDA(x_t) * ACh_t: write only when the postsynaptic neuron is depolarized (NMDA gate) and ACh is high.
    """

    def __init__(self, d, n_heads, tau_min=4.0, tau_max=256.0):
        super().__init__()
        assert d % n_heads == 0
        self.h, self.dh = n_heads, d // n_heads
        self.qkv = nn.Linear(d, 3 * d, bias=False)
        self.nmda = nn.Linear(d, n_heads)  # coincidence-detector gate
        self.og = nn.Linear(d, d, bias=False)  # output gate (postsynaptic depolarization)
        self.out = nn.Linear(d, d, bias=False)
        self.onorm = DivisiveNorm(self.dh)
        tau = torch.exp(torch.linspace(math.log(tau_min), math.log(tau_max), n_heads))
        lam = 1.0 - 1.0 / tau
        self.lam_logit = nn.Parameter(torch.log(lam / (1 - lam)))  # learnable
        nn.init.constant_(self.nmda.bias, 2.0)

    def lam(self):
        return torch.sigmoid(self.lam_logit)  # [h]

    def _proj(self, x, ach):
        B, T, _ = x.shape
        q, k, v = self.qkv(x).view(B, T, 3, self.h, self.dh).unbind(2)  # [B,T,h,dh]
        q = q * self.dh ** -0.5
        k = F.normalize(k, dim=-1)
        w = torch.sigmoid(self.nmda(x))  # [B,T,h]
        if ach is not None:
            w = w * ach
        return q, k, v, w

    def forward(self, x, ach=None):
        B, T, _ = x.shape
        q, k, v, w = self._proj(x, ach)
        idx = torch.arange(T, device=x.device)
        dist = (idx[:, None] - idx[None, :]).float()  # i - j
        causal = dist >= 0
        loglam = torch.log(self.lam())[:, None, None]  # [h,1,1]
        D = torch.where(causal, torch.exp(loglam * dist.clamp(min=0)), torch.zeros((), device=x.device))
        A = torch.einsum("bihd,bjhd->bhij", q, k) * D  # [B,h,T,T]
        A = A * w.permute(0, 2, 1)[:, :, None, :]  # write strength of each token j
        o = torch.einsum("bhij,bjhd->bihd", A, v)
        return self._finish(x, o)

    def _finish(self, x, o):
        B, T = x.shape[:2]
        o = self.onorm(o).reshape(B, T, -1)
        return self.out(o * F.silu(self.og(x)))

    def chunked(self, x, ach, S, C=128):
        """Read a sequence in chunks of C tokens: parallel inside a chunk, state S carried between chunks.
        Same result as forward() but O(n*C) instead of O(n^2); also returns the final state."""
        B, T, _ = x.shape
        q, k, v, w = self._proj(x, ach)
        lam = self.lam()  # [h]
        loglam = torch.log(lam)
        outs = []
        for s0 in range(0, T, C):
            qc, kc, vc, wc = q[:, s0:s0 + C], k[:, s0:s0 + C], v[:, s0:s0 + C], w[:, s0:s0 + C]
            n = qc.shape[1]
            idx = torch.arange(n, device=x.device)
            dist = (idx[:, None] - idx[None, :]).float()
            D = torch.where(dist >= 0, torch.exp(loglam[:, None, None] * dist.clamp(min=0)), torch.zeros((), device=x.device))
            A = torch.einsum("bihd,bjhd->bhij", qc, kc) * D * wc.permute(0, 2, 1)[:, :, None, :]
            o = torch.einsum("bhij,bjhd->bihd", A, vc)  # within-chunk part
            decay_in = torch.exp(loglam[None, :] * (idx[:, None] + 1).float())  # lam^(a+1) [n,h]
            o = o + torch.einsum("bhde,bihe->bihd", S.to(qc.dtype), qc) * decay_in[None, :, :, None].to(qc.dtype)
            decay_out = torch.exp(loglam[None, :] * (n - 1 - idx)[:, None].float())  # lam^(n-1-b) [n,h]
            wv = vc * (wc * decay_out[None].to(wc.dtype))[..., None]
            S = lam[None, :, None, None] ** n * S + torch.einsum("bjhd,bjhe->bhde", wv.float(), kc.float())
            outs.append(o)
        return self._finish(x, torch.cat(outs, 1)), S

    def step(self, x, ach, S):
        """One-token inference: constant-size memory S [B,h,dh,dh], no KV cache."""
        q, k, v, w = self._proj(x, ach)  # x [B,1,d]
        q, k, v, w = q[:, 0], k[:, 0], v[:, 0], w[:, 0]
        lam = self.lam()[None, :, None, None]
        S = lam * S + w[..., None, None] * torch.einsum("bhd,bhe->bhde", v, k)
        o = torch.einsum("bhde,bhe->bhd", S, q)[:, None]
        return self._finish(x, o), S


# ---------------------------------------------------------------- L5: spiking dendritic neurons
class DendriticSpikingLayer(nn.Module):
    """
    E population with dendrites + inhibitory I population, run for T gamma ticks per token.
      branch input    c_b = W_b x                (synapses on each dendritic branch)
      soma input      I = sum_b a_b * g(c_b)     (g: local branch nonlinearity, a_b >= 0)
      E potential     u <- beta*u + gain*I - W_IE s_I
      E spikes        s_E = H(u - (theta + adapt)); soft reset u -= theta*s_E; adapt <- rho*adapt + gamma*s_E
      I population    ui <- beta_i*ui + W_EI s_E;  s_I = H(ui - theta_I)
    Dale's law: W_EI, W_IE >= 0 (softplus). Output = E firing rate -> read out into the residual stream.
    """

    def __init__(self, d, n_e, branches=4, T=4, inh_ratio=0.25, theta=1.0, spike_in=False):
        super().__init__()
        self.spike_in = spike_in
        if spike_in:  # encode the input as ON/OFF spikes (like retinal cells) -> dendritic synapses only add
            self.theta_in_raw = nn.Parameter(torch.full((d,), 1.0))
        self.ne, self.B, self.T = n_e, branches, T
        self.ni = max(1, int(n_e * inh_ratio))  # inhibitory neurons are 20% of the total
        self.dend = nn.Linear(d, n_e * branches)
        self.branch_raw = nn.Parameter(torch.zeros(n_e, branches))  # softplus(0)=0.69
        self.beta_raw = nn.Parameter(torch.full((n_e,), 1.5))  # membrane leak, sigmoid ~ 0.82
        self.w_ei_raw = nn.Parameter(torch.randn(n_e, self.ni) * 0.1 - 4.0)
        self.w_ie_raw = nn.Parameter(torch.randn(self.ni, n_e) * 0.1 - 3.0)
        self.theta, self.theta_i = theta, 1.0
        self.rho, self.gamma = 0.5, 0.5  # refractory period / threshold adaptation
        self.out = nn.Linear(n_e, d, bias=False)
        nn.init.normal_(self.out.weight, std=0.02)

    def encode(self, x):
        th = F.softplus(self.theta_in_raw) + 0.1
        up, un = torch.zeros_like(x), torch.zeros_like(x)
        cp, cn = torch.zeros_like(x), torch.zeros_like(x)
        for _ in range(self.T):
            up = up + x
            un = un - x
            sp, sn = spike(up - th), spike(un - th)
            up, un = up - th * sp, un - th * sn
            cp, cn = cp + sp, cn + sn
        self.last_in_spikes = (cp + cn).sum(-1).mean().detach()  # input spikes per token
        return (cp - cn) / self.T * th  # quantized value, same scale

    def forward(self, x, gain=None):
        Bt = x.shape[:-1]
        if self.spike_in:
            x = self.encode(x)
        c = self.dend(x).view(*Bt, self.ne, self.B)
        I = (F.softplus(self.branch_raw) * F.silu(c)).sum(-1)  # spatial integration over dendrites
        if gain is not None:
            I = I * gain
        beta = torch.sigmoid(self.beta_raw)
        w_ei = F.softplus(self.w_ei_raw)
        w_ie = F.softplus(self.w_ie_raw)
        u = torch.zeros_like(I)
        adapt = torch.zeros_like(I)
        ui = torch.zeros(*Bt, self.ni, device=x.device, dtype=x.dtype)
        s_i = torch.zeros_like(ui)
        count = torch.zeros_like(I)
        count_i = torch.zeros_like(ui)
        for _ in range(self.T):  # T gamma ticks
            u = beta * u + I - s_i @ w_ie
            s = spike(u - (self.theta + adapt))
            u = u - self.theta * s
            adapt = self.rho * adapt + self.gamma * s
            ui = 0.5 * ui + s @ w_ei
            s_i = spike(ui - self.theta_i)
            ui = ui - self.theta_i * s_i
            count = count + s
            count_i = count_i + s_i
        rate = count / self.T
        self.last_rate = rate.mean()
        self.last_rate_i = (count_i / self.T).mean().detach()
        return self.out(rate)


# ---------------------------------------------------------------- one cortical region (6-layer column)
class CorticalRegion(nn.Module):
    """L4 input (divisive inhibition) -> L2/3 mixing over time -> L5 spiking output -> L6 receives feedback."""

    def __init__(self, d, n_heads, n_e, branches, T, spike_in=False):
        super().__init__()
        self.l4a = DivisiveNorm(d)
        self.l23 = HebbianSTDP(d, n_heads)
        self.l4b = DivisiveNorm(d)
        self.l5 = DendriticSpikingLayer(d, n_e, branches, T, spike_in=spike_in)
        self.l6_norm = DivisiveNorm(d)
        self.l6 = nn.Linear(d, d, bias=False)  # receives predictions from the region above
        nn.init.zeros_(self.l6.weight)

    def topdown(self, h_above):
        return self.l6(self.l6_norm(h_above))

    def forward(self, h, gain=None, ach=None, fb=None):
        if fb is not None:
            h = h + fb
        h = h + self.l23(self.l4a(h), ach)
        h = h + self.l5(self.l4b(h), gain)
        return h

    def forward_chunked(self, h, gain, ach, fb, S, C=128):
        if fb is not None:
            h = h + fb
        o, S = self.l23.chunked(self.l4a(h), ach, S, C)
        h = h + o
        h = h + self.l5(self.l4b(h), gain)
        return h, S

    def step(self, h, gain, ach, fb, S):
        if fb is not None:
            h = h + fb
        o, S = self.l23.step(self.l4a(h), ach, S)
        h = h + o
        h = h + self.l5(self.l4b(h), gain)
        return h, S


class Neuromodulators(nn.Module):
    """'Brainstem': from a baseline level (causal EMA of the input) emits noradrenaline (gain) and acetylcholine (write gate)."""

    def __init__(self, d, n_regions, n_heads, decay=0.9):
        super().__init__()
        self.L, self.h, self.decay = n_regions, n_heads, decay
        self.norm = DivisiveNorm(d)
        self.proj = nn.Linear(d, n_regions * (1 + n_heads))
        nn.init.zeros_(self.proj.weight)
        nn.init.zeros_(self.proj.bias)  # start: gain = 1, ach = 1

    def _split(self, m):
        m = 2 * torch.sigmoid(self.proj(self.norm(m)))
        m = m.view(*m.shape[:-1], self.L, 1 + self.h)
        return m[..., 0:1], m[..., 1:]  # gain [..,L,1], ach [..,L,h]

    def forward(self, x, K=256):
        # causal EMA as a 1-D convolution (kernel truncated at K steps: decay^K ~ 1e-12), O(n*K) instead of O(n^2)
        B, T, d = x.shape
        K = min(K, T)
        ker = (self.decay ** torch.arange(K - 1, -1, -1, device=x.device, dtype=torch.float32)).to(x.dtype)
        xt = F.pad(x.transpose(1, 2), (K - 1, 0))
        ema = F.conv1d(xt, ker.view(1, 1, K).expand(d, 1, K).contiguous(), groups=d).transpose(1, 2)
        t = torch.arange(T, device=x.device, dtype=torch.float32)
        norm = (1 - self.decay ** (t + 1).clamp(max=K)) / (1 - self.decay)
        return self._split(ema / norm[None, :, None].to(ema.dtype))

    def step(self, x, m_state, n_seen):
        # normalized EMA, identical to the parallel form
        m_state = self.decay * m_state + x
        norm = (1 - self.decay ** (n_seen + 1)) / (1 - self.decay)
        return self._split(m_state / norm), m_state


class DENDRA(nn.Module):
    def __init__(self, vocab, d=128, n_regions=4, n_heads=4, n_e=192, branches=4, T=4,
                 fb_cycles=1, use_neuromod=True, reflex=True, spike_in=False):
        super().__init__()
        self.emb = nn.Embedding(vocab, d)  # sensory encoding
        nn.init.normal_(self.emb.weight, std=0.02)
        self.regions = nn.ModuleList([CorticalRegion(d, n_heads, n_e, branches, T, spike_in) for _ in range(n_regions)])
        self.nm = Neuromodulators(d, n_regions, n_heads) if use_neuromod else None
        self.fb_cycles = fb_cycles
        self.norm_out = DivisiveNorm(d)
        self.reflex = reflex
        if reflex:
            self.norm_reflex = DivisiveNorm(d)
        self.n_heads = n_heads

    def _head(self, h, norm):
        return norm(h) @ self.emb.weight.t()

    def forward(self, idx):
        x = self.emb(idx)
        gain = ach = None
        if self.nm is not None:
            gain, ach = self.nm(x)
        prev = None
        for c in range(self.fb_cycles):
            h, outs = x, []
            for l, reg in enumerate(self.regions):
                fb = None
                if prev is not None and l + 1 < len(self.regions):
                    fb = reg.topdown(prev[l + 1])  # prediction from the region above in the previous pass
                h = reg(h, None if gain is None else gain[:, :, l], None if ach is None else ach[:, :, l], fb)
                outs.append(h)
            prev = outs
        logits = self._head(h, self.norm_out)
        reflex_logits = self._head(prev[0], self.norm_reflex) if self.reflex else None
        return logits, reflex_logits

    def spike_rates(self):
        return [r.l5.last_rate for r in self.regions]

    def init_state(self, B, dev):
        d = self.emb.weight.shape[1]
        L, h = len(self.regions), self.n_heads
        dh = d // h
        return {"S": [[torch.zeros(B, h, dh, dh, device=dev) for _ in range(L)] for _ in range(self.fb_cycles)],
                "m": torch.zeros(B, 1, d, device=dev), "t": 0}

    @torch.no_grad()
    def prefill(self, idx, C=128):
        """Read the whole prompt once (in chunks); return the last-token logits + state for further generation."""
        B, T = idx.shape
        st = self.init_state(B, idx.device)
        x = self.emb(idx)
        gain = ach = None
        if self.nm is not None:
            gain, ach = self.nm(x)
            m = torch.zeros(B, 1, x.shape[-1], device=x.device, dtype=torch.float32)
            dec = self.nm.decay ** torch.arange(T - 1, -1, -1, device=x.device, dtype=torch.float32)
            st["m"] = (x.float() * dec[None, :, None]).sum(1, keepdim=True)  # unnormalized EMA sum
        st["t"] = T
        L = len(self.regions)
        prev = None
        for c in range(self.fb_cycles):
            h, outs = x, []
            for l, reg in enumerate(self.regions):
                fb = reg.topdown(prev[l + 1]) if (prev is not None and l + 1 < L) else None
                h, st["S"][c][l] = reg.forward_chunked(h, None if gain is None else gain[:, :, l],
                                                       None if ach is None else ach[:, :, l], fb, st["S"][c][l], C)
                outs.append(h)
            prev = outs
        return self._head(h[:, -1:], self.norm_out), st

    @torch.no_grad()
    def step_token(self, tok, st):
        """Generate 1 token using only the constant-size synaptic state (O(1) memory per token)."""
        L = len(self.regions)
        x = self.emb(tok)
        gain = ach = None
        if self.nm is not None:
            (gain, ach), st["m"] = self.nm.step(x, st["m"], st["t"])
        st["t"] += 1
        prev = None
        for c in range(self.fb_cycles):
            hcur, outs = x, []
            for l, reg in enumerate(self.regions):
                fb = reg.topdown(prev[l + 1]) if (prev is not None and l + 1 < L) else None
                hcur, st["S"][c][l] = reg.step(hcur, None if gain is None else gain[:, :, l],
                                               None if ach is None else ach[:, :, l], fb, st["S"][c][l])
                outs.append(hcur)
            prev = outs
        return self._head(hcur, self.norm_out), st

    @torch.no_grad()
    def forward_recurrent(self, idx):
        """Run token by token with the constant-size state (to check it matches the parallel forward)."""
        st = self.init_state(idx.shape[0], idx.device)
        outs = []
        for t in range(idx.shape[1]):
            lg, st = self.step_token(idx[:, t:t + 1], st)
            outs.append(lg)
        return torch.cat(outs, 1)


# ---------------------------------------------------------------- GPT baseline (standard causal attention)
class GPTBlock(nn.Module):
    def __init__(self, d, n_heads):
        super().__init__()
        self.h = n_heads
        self.ln1, self.ln2 = nn.LayerNorm(d), nn.LayerNorm(d)
        self.qkv = nn.Linear(d, 3 * d, bias=False)
        self.proj = nn.Linear(d, d, bias=False)
        self.mlp = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Linear(4 * d, d))

    def forward(self, x):
        B, T, D = x.shape
        q, k, v = self.qkv(self.ln1(x)).view(B, T, 3, self.h, D // self.h).permute(2, 0, 3, 1, 4)
        a = F.scaled_dot_product_attention(q, k, v, is_causal=True).transpose(1, 2).reshape(B, T, D)
        x = x + self.proj(a)
        return x + self.mlp(self.ln2(x))


class GPT(nn.Module):
    def __init__(self, vocab, d=384, n_layers=6, n_heads=6, ctx=256):
        super().__init__()
        self.emb = nn.Embedding(vocab, d)
        self.pos = nn.Embedding(ctx, d)
        nn.init.normal_(self.emb.weight, std=0.02)
        nn.init.normal_(self.pos.weight, std=0.02)
        self.blocks = nn.ModuleList([GPTBlock(d, n_heads) for _ in range(n_layers)])
        self.ln = nn.LayerNorm(d)

    def forward(self, idx):
        x = self.emb(idx) + self.pos(torch.arange(idx.shape[1], device=idx.device))
        for b in self.blocks:
            x = b(x)
        return self.ln(x) @ self.emb.weight.t(), None

    def _run(self, x, cache, causal):
        for i, b in enumerate(self.blocks):
            B, T, D = x.shape
            q, k, v = b.qkv(b.ln1(x)).view(B, T, 3, b.h, D // b.h).permute(2, 0, 3, 1, 4)
            if cache[i] is not None:
                k = torch.cat([cache[i][0], k], 2); v = torch.cat([cache[i][1], v], 2)
            cache[i] = (k, v)
            a = F.scaled_dot_product_attention(q, k, v, is_causal=causal).transpose(1, 2).reshape(B, T, D)
            x = x + b.proj(a)
            x = x + b.mlp(b.ln2(x))
        return self.ln(x) @ self.emb.weight.t(), cache

    @torch.no_grad()
    def prefill(self, idx):
        """Read the prompt and store the KV cache (memory grows with context length)."""
        x = self.emb(idx) + self.pos(torch.arange(idx.shape[1], device=idx.device))
        lg, cache = self._run(x, [None] * len(self.blocks), True)
        return lg[:, -1:], {"kv": cache, "t": idx.shape[1]}

    @torch.no_grad()
    def step_token(self, tok, st):
        pos = min(st["t"], self.pos.num_embeddings - 1)
        x = self.emb(tok) + self.pos(torch.tensor([pos], device=tok.device))
        lg, st["kv"] = self._run(x, st["kv"], False)
        st["t"] += 1
        return lg, st

print("DENDRA and GPT defined")

### Fast decoder (used for text generation)

Each DENDRA generation step launches hundreds of tiny operations (4 ticks x E/I microcircuit x 6 regions). On a GPU the CPU has to issue them one by one, so a step takes ~20 ms while the GPU is mostly idle. Fixes:

1. **CUDA Graph:** capture the whole generation step once, then replay it with a single launch.
2. **Fused projections:** q, k, v, the NMDA gate and the output gate become one matrix multiply; softplus/sigmoid of parameters are precomputed.
3. **`torch.compile` (optional):** fuse the element-wise chain of the LIF loop into fewer kernels.

GPT gets exactly the same treatment (preallocated KV cache + CUDA Graph) for a fair comparison. Outputs match the original path (error < 1e-5).

In [ ]:
# Fast decoder (same code as dendra/fast.py)
"""Fast decoder for DENDRA and GPT: static state + CUDA Graph (+ optional torch.compile).

    from dendra.fast import make_decoder
    logits, dec = make_decoder(model, prompt_ids)          # prefill + capture a CUDA Graph (if a GPU is present)
    next_logits = dec(next_token_ids)                       # each call = one generation step

Why: one DENDRA generation step launches hundreds of tiny kernels (4 ticks x E/I microcircuit x regions),
so on a GPU the CPU-side launch overhead dominates (~20 ms/step regardless of context length). Fixes:
  1) Static state tensors -> the whole step is captured once as a CUDA Graph and replayed with one launch.
  2) q, k, v + NMDA gate + output gate fused into one matmul; softplus/sigmoid of parameters precomputed.
  3) Optional torch.compile to fuse the element-wise chain of the LIF loop into fewer kernels.
GPT gets the same treatment (preallocated KV cache + CUDA Graph) for a fair comparison.
"""

class FastDendra(nn.Module):
    """Inference version of DENDRA: same weights, same outputs, the state is a fixed-size tensor."""

    def __init__(self, m):
        super().__init__()
        assert not any(r.l5.spike_in for r in m.regions), "FastDendra does not support spike_in yet"
        self.m = m.eval()
        self.L, self.h = len(m.regions), m.n_heads
        self.d = m.emb.weight.shape[1]
        self.dh = self.d // self.h
        self.C = m.fb_cycles
        with torch.no_grad():
            self.Wp, self.bp, self.lam, self.branch, self.beta, self.wei, self.wie = [], [], [], [], [], [], []
            for r in m.regions:
                a = r.l23
                W = torch.cat([a.qkv.weight, a.nmda.weight, a.og.weight], 0)        # 1 matmul instead of 3
                b = torch.cat([torch.zeros(3 * self.d, device=W.device), a.nmda.bias,
                               torch.zeros(self.d, device=W.device)])
                self.Wp.append(W); self.bp.append(b)
                self.lam.append(a.lam().view(1, self.h, 1, 1))
                self.branch.append(F.softplus(r.l5.branch_raw))
                self.beta.append(torch.sigmoid(r.l5.beta_raw))
                self.wei.append(F.softplus(r.l5.w_ei_raw))
                self.wie.append(F.softplus(r.l5.w_ie_raw))

    def init_state(self, B, dev):
        S = torch.zeros(self.C, self.L, B, self.h, self.dh, self.dh, device=dev)
        return S, torch.zeros(B, self.d, device=dev), torch.zeros((), device=dev)

    @torch.no_grad()
    def from_prefill(self, st):
        S = torch.stack([torch.stack(row) for row in st["S"]]).float()
        return S, st["m"][:, 0].float(), torch.tensor(float(st["t"]), device=S.device)

    def _region(self, l, h, gain, ach, S):
        r, d, H, dh = self.m.regions[l], self.d, self.h, self.dh
        x = r.l4a(h)
        P = F.linear(x, self.Wp[l], self.bp[l])
        q = P[:, :d].view(-1, H, dh) * dh ** -0.5
        k = F.normalize(P[:, d:2 * d].view(-1, H, dh), dim=-1)
        v = P[:, 2 * d:3 * d].view(-1, H, dh)
        w = torch.sigmoid(P[:, 3 * d:3 * d + H])
        if ach is not None:
            w = w * ach
        og = P[:, 3 * d + H:]
        S = self.lam[l] * S + (w[..., None, None] * v[..., :, None] * k[..., None, :]).to(S.dtype)  # Hebb
        o = torch.einsum("bhde,bhe->bhd", S.to(q.dtype), q)
        o = r.l23.onorm(o).reshape(-1, d)
        h = h + r.l23.out(o * F.silu(og))
        l5 = r.l5
        c = l5.dend(r.l4b(h)).view(-1, l5.ne, l5.B)
        I = (self.branch[l] * F.silu(c)).sum(-1)
        if gain is not None:
            I = I * gain
        u = torch.zeros_like(I); ad = torch.zeros_like(I); cnt = torch.zeros_like(I)
        ui = torch.zeros(I.shape[0], l5.ni, device=I.device, dtype=I.dtype); si = torch.zeros_like(ui)
        beta, wie, wei = self.beta[l].to(I.dtype), self.wie[l].to(I.dtype), self.wei[l].to(I.dtype)
        for _ in range(l5.T):                                   # T gamma ticks, E/I microcircuit
            u = beta * u + I - si @ wie
            s = (u > l5.theta + ad).to(I.dtype)
            u = u - l5.theta * s
            ad = l5.rho * ad + l5.gamma * s
            ui = 0.5 * ui + s @ wei
            si = (ui > l5.theta_i).to(I.dtype)
            ui = ui - l5.theta_i * si
            cnt = cnt + s
        return h + l5.out(cnt / l5.T), S

    def step(self, tok, S, mstate, t):
        """tok [B]; S [C,L,B,h,dh,dh]; mstate [B,d]; t = number of tokens read so far (tensor).
        Updates S, mstate, t IN PLACE (so the CUDA Graph can reuse the same addresses) and returns logits."""
        m = self.m
        x = m.emb(tok)
        gain = ach = None
        if m.nm is not None:
            mnew = m.nm.decay * mstate + x.float()
            norm = (1 - torch.pow(torch.full((), m.nm.decay, device=x.device), t + 1)) / (1 - m.nm.decay)
            z = 2 * torch.sigmoid(m.nm.proj(m.nm.norm((mnew / norm).to(x.dtype))))
            z = z.view(-1, self.L, 1 + self.h)
            gain, ach = z[..., :1], z[..., 1:]
        newS, prev = [], None
        for c in range(self.C):
            hcur, outs, row = x, [], []
            for l in range(self.L):
                if prev is not None and l + 1 < self.L:
                    hcur = hcur + m.regions[l].topdown(prev[l + 1])
                hcur, Sl = self._region(l, hcur, None if gain is None else gain[:, l],
                                        None if ach is None else ach[:, l], S[c, l])
                outs.append(hcur); row.append(Sl)
            prev = outs
            newS.append(torch.stack(row))
        logits = m.norm_out(hcur) @ m.emb.weight.t()
        S.copy_(torch.stack(newS))
        if m.nm is not None:
            mstate.copy_(mnew)
        t.add_(1)
        return logits


class FastGPT(nn.Module):
    """GPT with a preallocated KV cache (no torch.cat per step) so it can run inside a CUDA Graph."""

    def __init__(self, g, maxlen):
        super().__init__()
        self.g, self.maxlen = g.eval(), maxlen
        self.L, self.H = len(g.blocks), g.blocks[0].h
        self.d = g.emb.weight.shape[1]

    def init_state(self, B, dev, dtype=torch.float32):
        return torch.zeros(self.L, 2, B, self.H, self.maxlen, self.d // self.H, device=dev, dtype=dtype)

    @torch.no_grad()
    def from_prefill(self, st, dtype=torch.float32):
        kv = st["kv"]; n = kv[0][0].shape[2]; B = kv[0][0].shape[0]
        cache = self.init_state(B, kv[0][0].device, dtype)
        for i, (k, v) in enumerate(kv):
            cache[i, 0, :, :, :n] = k; cache[i, 1, :, :, :n] = v
        return cache, torch.tensor(n, device=cache.device)

    def step(self, tok, cache, pos):
        g = self.g
        x = g.emb(tok) + g.pos(pos.clamp(max=g.pos.num_embeddings - 1).view(1))      # [B, D]
        invalid = (torch.arange(self.maxlen, device=x.device) > pos).view(1, 1, -1)  # positions not written yet
        for i, b in enumerate(g.blocks):
            B, D = x.shape
            H, dh = b.h, D // b.h
            qkv = b.qkv(b.ln1(x)).view(B, 3, H, dh)
            q, k, v = qkv[:, 0], qkv[:, 1], qkv[:, 2]                                # [B, H, dh]
            cache[i, 0].index_copy_(2, pos.view(1), k[:, :, None].to(cache.dtype))   # write K/V in place
            cache[i, 1].index_copy_(2, pos.view(1), v[:, :, None].to(cache.dtype))
            att = torch.einsum("bhd,bhnd->bhn", q, cache[i, 0].to(q.dtype)).float() * dh ** -0.5
            att = att.masked_fill(invalid, float("-inf")).softmax(-1).to(q.dtype)
            a = torch.einsum("bhn,bhnd->bhd", att, cache[i, 1].to(q.dtype)).reshape(B, D)
            x = x + b.proj(a)
            x = x + b.mlp(b.ln2(x))
        pos.add_(1)
        return g.ln(x) @ g.emb.weight.t()


class Graphed:
    """Capture one generation step as a CUDA Graph and replay it with a single launch. Without a GPU it runs eagerly.
    fn(tok, *state) must update the state in place and return the logits."""

    def __init__(self, fn, tok, state, use_graph=True, use_compile=False):
        self.fn, self.state, self.graph = fn, state, None
        if use_compile:
            try:
                self.fn = torch.compile(fn, mode="max-autotune-no-cudagraphs", dynamic=False)
            except Exception as e:
                print("torch.compile failed, skipping:", type(e).__name__, str(e)[:120])
        if use_graph and tok.device.type == "cuda":
            self.tok = tok.clone()
            backup = [x.clone() for x in state]
            s = torch.cuda.Stream(); s.wait_stream(torch.cuda.current_stream())
            with torch.cuda.stream(s), torch.no_grad(), autocast():
                for _ in range(3):                      # warm-up (and compilation if compile is on)
                    self.fn(self.tok, *state)
            torch.cuda.current_stream().wait_stream(s)
            try:
                self.graph = torch.cuda.CUDAGraph()
                with torch.cuda.graph(self.graph), torch.no_grad(), autocast():
                    self.out = self.fn(self.tok, *state)
            except Exception as e:                     # cannot capture -> run eagerly instead of crashing
                self.graph = None
                print(f"  (CUDA Graph unavailable for {type(fn.__self__).__name__}: "
                      f"{type(e).__name__}; running eagerly)")
                try:
                    torch.cuda.synchronize()
                except Exception:
                    pass
            for x, b in zip(state, backup):             # restore the state to right after prefill
                x.copy_(b)

    def __call__(self, tok):
        if self.graph is None:
            with torch.no_grad(), autocast():
                return self.fn(tok, *self.state)
        self.tok.copy_(tok)
        self.graph.replay()
        return self.out


@torch.no_grad()
def make_decoder(model, prompt_ids, use_graph=True, use_compile=False, maxlen=None):
    """Read the prompt (prefill), then return (last-token logits, dec) where next_logits = dec(tok [B])."""
    with autocast():
        lg, st = model.prefill(prompt_ids)
    if isinstance(model, GPT):
        fm = FastGPT(model, maxlen or (prompt_ids.shape[1] + 256))
        state = list(fm.from_prefill(st, dtype=torch.float16 if AMP_DTYPE else torch.float32))
    else:
        fm = FastDendra(model)
        state = list(fm.from_prefill(st))
    tok0 = lg[:, -1].argmax(-1)
    runner = Graphed(fm.step, tok0, state, use_graph, use_compile)
    return lg[:, -1], runner

## 3. Training

In [ ]:
def make_model(kind):
    if kind == "dendra":
        return DENDRA(V, d=CFG["d"], n_regions=CFG["regions"], n_heads=CFG["heads"], n_e=CFG["n_e"],
                        branches=CFG["branches"], T=CFG["T"], fb_cycles=CFG["fb_cycles"],
                        use_neuromod=CFG["neuromod"], spike_in=CFG["spike_in"])
    return GPT(V, d=CFG["d"], n_layers=CFG["gpt_layers"], n_heads=CFG["heads"], ctx=CFG["ctx"])

def autocast():
    return torch.autocast("cuda", dtype=AMP_DTYPE) if AMP_DTYPE else torch.autocast("cpu", enabled=False)

@torch.no_grad()
def evaluate(model, kind, n_batches, dev=None):
    model.eval()
    g = torch.Generator().manual_seed(123)
    tot = cnt = tot_r = 0.0; conf, okr, okf, rates = [], [], [], []
    for _ in range(n_batches):
        x, y = get_batch(val_t, CFG["batch"], g, dev)
        with autocast():
            logits, rlog = model(x)
        logits = logits.float()
        tot += F.cross_entropy(logits.reshape(-1, V), y.reshape(-1), reduction="sum").item(); cnt += y.numel()
        okf.append((logits.argmax(-1) == y).flatten())
        if rlog is not None:
            rlog = rlog.float(); pr = rlog.softmax(-1)
            tot_r += F.cross_entropy(rlog.reshape(-1, V), y.reshape(-1), reduction="sum").item()
            conf.append(pr.max(-1).values.flatten()); okr.append((pr.argmax(-1) == y).flatten())
        if kind == "dendra":
            rates.append(torch.stack([r.float() for r in model.spike_rates()]).cpu())
    res = {"val_loss": tot / cnt, "ppl": math.exp(tot / cnt), "acc": torch.cat(okf).float().mean().item()}
    if conf:
        conf, okr, okf_ = torch.cat(conf), torch.cat(okr), torch.cat(okf)
        res["reflex_loss"] = tot_r / cnt
        res["reflex"] = [{"thr": t, "exit_frac": (conf > t).float().mean().item(),
                          "acc_exited": okr[conf > t].float().mean().item() if (conf > t).any() else float("nan"),
                          "acc_mixed": torch.where(conf > t, okr, okf_).float().mean().item()} for t in (0.5, 0.7, 0.9)]
    if rates:
        res["spike_rates"] = torch.stack(rates).mean(0).tolist()
    model.train()
    return res

import threading, traceback
STOP = threading.Event()   # shared stop signal for the training threads
HELPER = {"dev": None, "free": threading.Event()}   # GPU handed over when GPT finishes
COMPILE_LOCK = threading.Lock()

class LossWrap(nn.Module):
    # Compute the loss inside forward, so that when a batch is split over 2 GPUs (DataParallel)
    # each GPU computes the firing rate of its own replica before results are gathered.
    def __init__(self, m, kind):
        super().__init__(); self.m, self.kind = m, kind
    def forward(self, x, y):
        logits, rlog = self.m(x)
        ce = F.cross_entropy(logits.float().reshape(-1, logits.shape[-1]), y.reshape(-1))
        if self.kind != "dendra":
            return ce.view(1), ce.new_zeros(1), ce.new_zeros(1)
        rates = torch.stack(self.m.spike_rates()).float()
        homeo = ((rates - CFG["rate_target"]) ** 2).sum()            # homeostasis
        rce = F.cross_entropy(rlog.float().reshape(-1, rlog.shape[-1]), y.reshape(-1)) if rlog is not None else ce * 0
        return ce.view(1), homeo.view(1), rce.view(1)

def train(kind, dev=None):
    dev = dev or device
    if str(dev).startswith("cuda"):
        torch.cuda.set_device(dev)
    torch.manual_seed(CFG["seed"])
    model = make_model(kind).to(dev)
    n_params = sum(p.numel() for p in model.parameters())
    print(f"\n===== {kind} on {dev}: {n_params/1e6:.2f}M params =====", flush=True)
    decay = [p for n, p in model.named_parameters() if p.dim() >= 2]
    no_decay = [p for n, p in model.named_parameters() if p.dim() < 2]
    opt = torch.optim.AdamW([{"params": decay, "weight_decay": CFG["weight_decay"]},
                             {"params": no_decay, "weight_decay": 0.0}], lr=CFG["lr"], betas=(0.9, 0.95),
                            **({"fused": True} if str(dev).startswith("cuda") else {}))
    plan = {"total": CFG["steps"]}   # actual number of steps, adjusted to the time limit
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1, (s + 1) / CFG["warmup"]) *
                                              (0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * min(1, s / plan["total"])))))
    scaler = torch.amp.GradScaler("cuda", enabled=(AMP_DTYPE == torch.float16))
    hist = {"step": [], "train": [], "val": [], "time": []}
    t0 = time.time(); run = 0.0; step = 0; stopped = ""
    runner = LossWrap(model, kind); shared = None; compiled = False
    if CFG.get("compile"):
        try:
            runner = LossWrap(torch.compile(model), kind); compiled = True
        except Exception as e:
            print(f"[{kind}] torch.compile failed, using eager mode: {type(e).__name__}", flush=True)

    def train_step(x, y):
        with autocast():
            ce, homeo, rce = (o.mean() for o in runner(x, y))
            total = ce + (CFG["homeo_w"] * homeo + CFG["reflex_w"] * rce if kind == "dendra" else 0)
        opt.zero_grad(set_to_none=True)
        scaler.scale(total).backward()
        return ce

    def do_eval(step):
        ev = evaluate(model, kind, CFG["eval_batches"], dev)
        hist["step"].append(step); hist["train"].append(run); hist["val"].append(ev["val_loss"])
        hist["time"].append(time.time() - t0)
        msg = f"[{kind}] EVAL step {step}: val loss {ev['val_loss']:.3f} (ppl {ev['ppl']:.1f})"
        if kind == "dendra":
            msg += " | xung " + " ".join(f"{r:.2f}" for r in ev["spike_rates"])
        print(msg, flush=True)

    try:
        while step <= plan["total"]:
            if STOP.is_set():
                stopped = f"stopped manually at step {step}"; print(f"[{kind}] Stopped at step {step}.", flush=True); break
            # the other GPU is free (GPT finished) -> split each batch over 2 GPUs
            if (kind == "dendra" and shared is None and CFG.get("share_free_gpu") and HELPER["free"].is_set()
                    and HELPER["dev"] is not None and HELPER["dev"] != str(dev)):
                if compiled:   # a compiled model on 1 GPU is usually faster than an uncompiled one split over 2
                    shared = {"step": -10**9, "t": 0}
                    print(f"[{kind}] GPT finished; using torch.compile so staying on 1 GPU", flush=True)
                    continue
                if str(dev).startswith("cuda"):
                    runner = nn.DataParallel(LossWrap(model, kind), device_ids=[dev, HELPER["dev"]], output_device=dev)
                shared = {"step": step, "t": time.time()}
                print(f"[{kind}] GPT finished -> DENDRA also uses {HELPER['dev']} from step {step} (batch split in two)", flush=True)
            x, y = get_batch(train_t, CFG["batch"], dev=dev)
            if step < 2 and compiled:
                try:
                    with COMPILE_LOCK:     # compile one thread at a time (torch.compile is not thread-safe while compiling)
                        if step == 0: print(f"[{kind}] compiling (torch.compile), the first time takes 1-3 minutes...", flush=True)
                        loss = train_step(x, y)
                except Exception as e:
                    print(f"[{kind}] torch.compile failed ({type(e).__name__}: {str(e)[:120]}), falling back to eager mode", flush=True)
                    runner = LossWrap(model, kind); compiled = False
                    loss = train_step(x, y)
            else:
                loss = train_step(x, y)
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step()
            run = 0.98 * run + 0.02 * loss.item() if step else loss.item()
            if step == 0:
                do_eval(0)
            elif step == 2:
                t_train = time.time()      # start timing AFTER compilation (excludes waiting time)
            elif step == 22 and CFG.get("max_minutes"):
                sec = (time.time() - t_train) / 20
                budget = int(CFG["max_minutes"] * 60 / (sec * 1.1))   # keep 10% for evaluation
                plan["total"] = max(21, min(CFG["steps"], budget))
                print(f"[{kind}] speed {sec*1000:.0f} ms/step -> will run {plan['total']} steps "
                      f"(~{plan['total']*sec*1.1/60:.0f} min, limit {CFG['max_minutes']} min)", flush=True)
            if shared and shared["step"] >= 0 and step == shared["step"] + 20:
                sec = (time.time() - shared["t"]) / 20
                msg = f"[{kind}] speed with 2 GPUs: {sec*1000:.0f} ms/step"
                if CFG.get("max_minutes"):   # time left over -> run more steps (up to CFG["steps"])
                    left = CFG["max_minutes"] * 60 - (time.time() - t0)
                    plan["total"] = max(step + 1, min(CFG["steps"], step + int(left / (sec * 1.1))))
                    msg += f" -> {plan['total']} steps in total"
                print(msg, flush=True)
            if step and step % CFG["log_every"] == 0:
                el = time.time() - t0
                eta = (plan["total"] - step) * el / step
                print(f"[{kind}] step {step}/{plan['total']} | loss {run:.3f} | {el/60:.1f} min | ~{eta/60:.1f} min left", flush=True)
            if step and (step % CFG["eval_every"] == 0 or step == plan["total"]):
                do_eval(step)
            step += 1
    except KeyboardInterrupt:
        stopped = f"stopped manually at step {step}"
        print(f"\n[{kind}] Stopped at step {step}. Saving the results so far...", flush=True)
    final = evaluate(model, kind, CFG["eval_batches"] * 3, dev)
    final.update(params=n_params, train_time_s=time.time() - t0, steps_done=step, stopped=stopped, hist=hist)
    torch.save(model.state_dict(), f"{OUT}/{kind}.pt")
    print(f"[{kind}] DONE: val loss {final['val_loss']:.3f} (ppl {final['ppl']:.1f}), "
          f"{step} steps, {final['train_time_s']/60:.1f} min", flush=True)
    return model, final

results, models = {}, {}
STOP.clear()
kinds = ["dendra"] + (["gpt"] if CFG["compare_gpt"] else [])
n_gpu = torch.cuda.device_count() if device == "cuda" else 0
devs = ["cpu", "cpu"] if os.environ.get("DENDRA_FAKE2GPU") == "1" else [f"cuda:{i}" for i in range(n_gpu)]

if CFG["parallel_gpus"] and len(devs) >= 2 and len(kinds) == 2:
    # One model per GPU, trained in parallel with 2 threads.
    # Press Stop once: both models stop and save their results so far.
    print(f"Running in parallel: DENDRA on {devs[0]}, GPT on {devs[1]}")
    out = {}
    HELPER["dev"] = devs[1]; HELPER["free"].clear()
    def worker(kind, dev):
        try:
            out[kind] = train(kind, dev)
        except Exception as e:
            traceback.print_exc(); out[kind] = e
        finally:
            if kind == "gpt":                       # hand the GPU over to DENDRA
                if str(dev).startswith("cuda"): torch.cuda.empty_cache()
                HELPER["free"].set()
    threads = [threading.Thread(target=worker, args=(k, devs[i]), daemon=True) for i, k in enumerate(kinds)]
    for t in threads: t.start()
    try:
        while any(t.is_alive() for t in threads):
            time.sleep(0.5)
    except KeyboardInterrupt:
        print("\nStopping both models and saving results...", flush=True)
        STOP.set()
        for t in threads: t.join()
    for k in kinds:
        if isinstance(out.get(k), Exception): raise out[k]
        models[k], results[k] = out[k]
else:
    # One GPU: run one after the other. Press Stop once: the running model stops, saves, then the next one starts.
    for k in kinds:
        models[k], results[k] = train(k)
json.dump({"cfg": CFG, "data": DATA_NAME, "results": results}, open(f"{OUT}/results.json", "w"), indent=1, ensure_ascii=False)

## 4. Results

In [ ]:
import matplotlib.pyplot as plt
assert results, "No results yet: run the Training cell first."
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for k, r in results.items():
    h = r["hist"]
    ax[0].plot(h["step"], h["val"], label=f"{k} (val)")
    ax[0].plot(h["step"], h["train"], "--", alpha=0.5, label=f"{k} (train)")
    ax[1].plot(np.array(h["time"]) / 60, h["val"], label=k)
ax[0].set_xlabel("step"); ax[0].set_ylabel("loss (nats/token)"); ax[0].legend(); ax[0].set_title("Per step")
ax[1].set_xlabel("minutes"); ax[1].set_ylabel("val loss"); ax[1].legend(); ax[1].set_title("Per wall-clock time")
plt.tight_layout(); plt.savefig(f"{OUT}/loss.png", dpi=120); plt.show()

print(f"{'Model':8} {'Params':>9} {'Val loss':>9} {'PPL':>8} {'Acc':>7} {'Min':>6}")
for k, r in results.items():
    print(f"{k:8} {r['params']/1e6:8.2f}M {r['val_loss']:9.3f} {r['ppl']:8.1f} {r['acc']*100:6.1f}% {r['train_time_s']/60:6.1f}"
          f"  ({r['steps_done']} steps{', ' + r['stopped'] if r['stopped'] else ''})")

In [ ]:
assert "dendra" in results, "No DENDRA results yet: run the Training cell first."
r = results["dendra"]
print("E firing rate per region:", ", ".join(f"{x:.3f}" for x in r["spike_rates"]))
print("\nReflex path (early exit at region 1):")
print(f"{'Thresh':>7} {'Exited':>7} {'Acc of exited':>16} {'Overall acc':>12}   (no exit: {r['acc']*100:.1f}%)")
for e in r["reflex"]:
    print(f"{e['thr']:7.1f} {e['exit_frac']*100:6.1f}% {e['acc_exited']*100:15.1f}% {e['acc_mixed']*100:9.1f}%")

m = models["dendra"]
print("\nLearned synaptic time constants (tokens), per region:")
for i, reg in enumerate(m.regions):
    print(f"  region {i+1}:", [round(t, 1) for t in (1 / (1 - reg.l23.lam())).tolist()])

# Check: parallel mode (training) == token-by-token mode (inference, O(1) memory)
m.eval()
x, _ = get_batch(val_t, 2, dev=next(m.parameters()).device); x = x[:, :48]
with torch.no_grad():
    lp, _ = m(x); lr_ = m.forward_recurrent(x)
print("\nParallel vs recurrent max difference:", (lp.float() - lr_.float()).abs().max().item())

## 5. Text generation (fast mode)

* **GPT:** reads the prompt once and stores a **KV cache**; each new token only computes the new part, but the cache grows with the context.
* **DENDRA:** reads the prompt **in chunks** (O(n)), then generates each token from a **constant-size synaptic state**, with no cache.
* Both use the **fast decoder** (CUDA Graph) above.

In [ ]:
def sync():
    if device == "cuda": torch.cuda.synchronize()

@torch.no_grad()
def generate(model, prompt, n=200, temp=0.8, top_k=50):
    model.eval()
    dev = next(model.parameters()).device
    ids = tok.encode(prompt).ids
    if isinstance(model, GPT):              # GPT only knows the positions within the trained context
        n = max(0, min(n, CFG["ctx"] - len(ids)))
    x = torch.tensor([ids], device=dev)
    sync(); t0 = time.time()
    lg0, dec = make_decoder(model, x, use_graph=True, maxlen=len(ids) + n + 1)   # CUDA Graph when a GPU is present
    logits = lg0
    out = []
    for _ in range(n):
        lg = logits[0].float() / temp
        v, _ = torch.topk(lg, top_k); lg[lg < v[-1]] = -float("inf")
        nxt = torch.multinomial(lg.softmax(-1), 1)
        if nxt.item() == EOS: break
        out.append(nxt.item())
        logits = dec(nxt.view(1))
    sync(); dt = time.time() - t0
    return prompt + tok.decode(out), len(out) / max(dt, 1e-9)

prompts = ["Ngày xửa ngày xưa,", "Một hôm, cô bé"] if DATA_NAME.endswith("_vi") else ["Once upon a time", "ROMEO:"]   # Vietnamese: 'Once upon a time,' / 'One day, the little girl'
for k, mdl in models.items():
    for p in prompts:
        text, tps = generate(mdl, p)
        print(f"--- {k} | {tps:.0f} tokens/s\n{text}\n")

## 6. Response speed (no training)

Uses **randomly initialized models with the same configuration** (speed does not depend on the weights; GPT gets a position table long enough for long contexts).

* **Prefill:** time to read the whole prompt.
* **Decode:** mean time per new token when the context already holds n tokens.
* **Memory:** KV cache (GPT) or synaptic state (DENDRA) per conversation.

In [ ]:
import gc, math
torch._dynamo.config.cache_size_limit = 256
BENCH_CTX = [128, 512, 1024, 2048, 4096] if not QUICK_TEST else [32, 64]
BENCH_BATCH = [1, 16] if not QUICK_TEST else [1, 2]
COMPILE_CTX = [512, 4096] if not QUICK_TEST else [64]
N_DECODE = 32 if not QUICK_TEST else 4
gc.collect()
if device == "cuda": torch.cuda.empty_cache()

def state_mb(st):
    if "kv" in st:
        return sum(k.numel() * k.element_size() + v.numel() * v.element_size() for k, v in st["kv"]) / 1e6
    return sum(S.numel() * S.element_size() for row in st["S"] for S in row) / 1e6

def time_decode(fn, tok, n):
    fn(tok); sync(); t = time.time()
    for _ in range(n): fn(tok)
    sync(); return (time.time() - t) / n * 1000

def bench_model(kind):
    torch.manual_seed(0)
    if kind == "dendra":
        m = make_model("dendra")
    else:
        m = GPT(V, d=CFG["d"], n_layers=CFG["gpt_layers"], n_heads=CFG["heads"], ctx=max(BENCH_CTX) + N_DECODE + 16)
    m = m.to(device).eval()
    rows = []
    for b in BENCH_BATCH:
        for n in BENCH_CTX:
            x = torch.randint(0, V, (b, n), device=device)
            r = dict(model=kind, batch=b, ctx=n)
            try:
                with torch.no_grad(), autocast():
                    m.prefill(x[:, :min(n, 64)])
                    sync(); t = time.time(); lg, st = m.prefill(x); sync(); r["prefill_ms"] = (time.time() - t) * 1e3
                tok = lg[:, -1].argmax(-1)
                def old(tk):
                    with torch.no_grad(), autocast():
                        return m.step_token(tk[:, None], st)[0]
                r["old"] = time_decode(old, tok, N_DECODE)
                r["mem_mb_per_seq"] = state_mb(st) / b
                del st
                try:
                    _, dec = make_decoder(m, x, use_graph=True, maxlen=n + 2 * N_DECODE + 8)
                    r["fast"] = time_decode(dec, tok, N_DECODE); del dec
                except Exception as e:
                    print(f"  [{kind}] CUDA Graph error: {type(e).__name__}: {str(e)[:150]}"); r["fast"] = float("nan")
                if CFG["bench_compile"] and n in COMPILE_CTX:
                    torch._dynamo.reset()
                    try:
                        _, dec = make_decoder(m, x, use_graph=True, use_compile=True, maxlen=n + 2 * N_DECODE + 8)
                        r["fast+compile"] = time_decode(dec, tok, N_DECODE); del dec
                    except Exception as e:
                        print(f"  [{kind}] compile error: {type(e).__name__}: {str(e)[:150]}"); r["fast+compile"] = float("nan")
            except torch.cuda.OutOfMemoryError:
                print(f"  [{kind}] out of memory at batch {b}, context {n}"); torch.cuda.empty_cache()
            rows.append(r)
            best = min(v for k2, v in r.items() if k2 in ("old", "fast", "fast+compile") and not math.isnan(v))
            extra = f" | fast+compile {r['fast+compile']:6.2f}" if "fast+compile" in r else ""
            print(f"[{kind:6}] batch {b:2d} | context {n:5d} | decode: old {r.get('old', float('nan')):6.2f} | "
                  f"fast {r.get('fast', float('nan')):6.2f}{extra} ms/step -> {b / best * 1000:7.0f} tokens/s | "
                  f"prefill {r.get('prefill_ms', float('nan')):7.1f} ms | memory {r.get('mem_mb_per_seq', float('nan')):5.2f} MB", flush=True)
            gc.collect()
            if device == "cuda": torch.cuda.empty_cache()
    del m; gc.collect()
    if device == "cuda": torch.cuda.empty_cache()
    return rows

bench = bench_model("dendra") + bench_model("gpt")
json.dump(bench, open(f"{OUT}/bench.json", "w"), indent=1, ensure_ascii=False)

def best_ms(r):
    return min(v for k, v in r.items() if k in ("old", "fast", "fast+compile") and not math.isnan(v))

print("\nWHICH IS FASTER? (each model in its own fastest mode)")
for b in BENCH_BATCH:
    for n in BENCH_CTX:
        c = next(r for r in bench if r["model"] == "dendra" and r["batch"] == b and r["ctx"] == n)
        g = next(r for r in bench if r["model"] == "gpt" and r["batch"] == b and r["ctx"] == n)
        bc, bg = best_ms(c), best_ms(g)
        win = "DENDRA" if bc < bg else "GPT"
        print(f"  batch {b:2d}, context {n:5d}: DENDRA {bc:6.2f} ms | GPT {bg:6.2f} ms -> {win} faster by {max(bc, bg) / min(bc, bg):.2f}x")

fig, ax = plt.subplots(1, len(BENCH_BATCH) + 1, figsize=(6 * (len(BENCH_BATCH) + 1), 4))
for j, b in enumerate(BENCH_BATCH):
    for kind, c in [("dendra", "tab:blue"), ("gpt", "tab:gray")]:
        rs = [r for r in bench if r["model"] == kind and r["batch"] == b]
        ax[j].plot([r["ctx"] for r in rs], [r.get("old") for r in rs], "--", color=c, alpha=0.6, label=f"{kind} (old)")
        ax[j].plot([r["ctx"] for r in rs], [r.get("fast") for r in rs], "o-", color=c, label=f"{kind} (fast)")
        rc = [r for r in rs if "fast+compile" in r]
        ax[j].plot([r["ctx"] for r in rc], [r["fast+compile"] for r in rc], "*", ms=12, color=c, label=f"{kind} (fast+compile)")
    ax[j].set_title(f"Decode step, batch {b}"); ax[j].set_ylabel("ms / step"); ax[j].set_yscale("log")
for kind, c in [("dendra", "tab:blue"), ("gpt", "tab:gray")]:
    rs = [r for r in bench if r["model"] == kind and r["batch"] == BENCH_BATCH[0]]
    ax[-1].plot([r["ctx"] for r in rs], [r.get("mem_mb_per_seq") for r in rs], "o-", color=c, label=kind)
ax[-1].set_title("Memory per conversation"); ax[-1].set_ylabel("MB")
for a in ax:
    a.set_xscale("log", base=2); a.set_xlabel("context length (tokens)"); a.legend(fontsize=8)
plt.tight_layout(); plt.savefig(f"{OUT}/bench.png", dpi=120); plt.show()

## 7. What to try next

* **2 GPUs (T4 x2):** DENDRA and GPT train in parallel, one GPU each; when GPT finishes first, DENDRA uses both GPUs (`share_free_gpu`). Set `parallel_gpus=False` to run them one after the other.
* **Speed:** section 6 compares 3 modes (old / fast = CUDA Graph / fast + compile) for both models and prints "WHICH IS FASTER?" for every context length; batch 16 simulates serving many users at once.
* **Faster training:** `compile=True` (default) fuses the operations of the LIF loop. For an even lighter DENDRA (requires retraining): reduce `T` from 4 to 2, `branches` from 4 to 2, or `n_e`.
* **Fair budget:** DENDRA is slower than GPT per step on GPU (the T-tick loop). To compare at equal wall-clock time, see the right-hand plot or increase `steps` for GPT.
* **Feedback:** `fb_cycles=2`. **Spiking input:** `spike_in=True`. **No neuromodulators:** `neuromod=False` to measure their contribution.
* **Bigger model:** increase `d`, `regions`, `n_e`; reduce `batch` if the GPU runs out of memory. A T4 has 16 GB.
* **Time:** a Kaggle GPU session lasts up to ~12 hours; checkpoints are saved to `/kaggle/working/*.pt`. Use *Save Version -> Save & Run All* to run in the background and keep the results.